In [41]:
from pyspark.sql.session import SparkSession

def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .config("spark.ui.port","4141")
        .config("spark.port.maxRetries", "0")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

def stop_start(app_name):
   
    global spark 
    
    if 'spark' in globals():
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
             print(f"No active session to stop. Error: {e}")
    else:
        print("No existing 'spark' variable found.")

    # 3. Return the fresh session
    print("Starting new Spark session...")
    return get_spark_session(app_name)


spark = stop_start("working_with_different_data_type")

Old Spark session successfully stopped.
Starting new Spark session...


### Working with Dates and Timestamps

Dates and times are a constant challenge in programming languages and databases. It’s always
necessary to keep track of timezones and ensure that formats are correct and valid

There
are dates, which focus exclusively on calendar dates, and timestamps, which include both date
and time information

we often store our timestamps or dates as strings and convert them into date types at
runtime. 

In [42]:
spark.conf.get("spark.sql.sessionLocalTimeZone", "UTC")
spark.conf.set("spark.sql.sessionLocalTimeZone", "America/New_York")



"""
spark-submit \
  --conf spark.sql.sessionLocalTimeZone=UTC \
  your_script.py

  

  -- jvm level (Fallback)

--conf spark.driver.extraJavaOptions="-Duser.timezone=GMT" \
--conf spark.executor.extraJavaOptions="-Duser.timezone=GMT"

  """

'\nspark-submit   --conf spark.sql.sessionLocalTimeZone=UTC   your_script.py\n\n\n\n  -- jvm level (Fallback)\n\n--conf spark.driver.extraJavaOptions="-Duser.timezone=GMT" --conf spark.executor.extraJavaOptions="-Duser.timezone=GMT"\n\n  '

In [43]:
retail_df = (

    spark
    .read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load("../data/retail-data/")
)

In [44]:
retail_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [45]:
retail_df.show(5, False)

+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+
|537226   |22811    |SET OF 6 T-LIGHTS CACTI       |6       |2010-12-06 08:34:00|2.95     |15987.0   |United Kingdom|
|537226   |21713    |CITRONELLA CANDLE FLOWERPOT   |8       |2010-12-06 08:34:00|2.1      |15987.0   |United Kingdom|
|537226   |22927    |GREEN GIANT GARDEN THERMOMETER|2       |2010-12-06 08:34:00|5.95     |15987.0   |United Kingdom|
|537226   |20802    |SMALL GLASS SUNDAE DISH CLEAR |6       |2010-12-06 08:34:00|1.65     |15987.0   |United Kingdom|
|537226   |22052    |VINTAGE CARAVAN GIFT WRAP     |25      |2010-12-06 08:34:00|0.42     |15987.0   |United Kingdom|
+---------+---------+------------------------------+----

In [46]:
from pyspark.sql.functions import current_date, current_timestamp, current_time, current_timezone

df = spark.sparkContext.parallelize([(1,), (2,), (3,), (4,), (5,) ]).toDF(["number",])

df.show()

+------+
|number|
+------+
|     1|
|     2|
|     3|
|     4|
|     5|
+------+



In [47]:
sample_date_df = (

    df
    .withColumn("current_data", current_date())
    .withColumn("current_time", current_time())
    .withColumn("current_timezone", current_timezone())
    .withColumn("current_timestamp", current_timestamp())
    # .show(5,False)
)

sample_date_df.show(10,False)


+------+------------+---------------+----------------+--------------------------+
|number|current_data|current_time   |current_timezone|current_timestamp         |
+------+------------+---------------+----------------+--------------------------+
|1     |2026-11-08  |16:57:57.730344|Asia/Kolkata    |2026-11-08 16:57:57.730344|
|2     |2026-11-08  |16:57:57.730344|Asia/Kolkata    |2026-11-08 16:57:57.730344|
|3     |2026-11-08  |16:57:57.730344|Asia/Kolkata    |2026-11-08 16:57:57.730344|
|4     |2026-11-08  |16:57:57.730344|Asia/Kolkata    |2026-11-08 16:57:57.730344|
|5     |2026-11-08  |16:57:57.730344|Asia/Kolkata    |2026-11-08 16:57:57.730344|
+------+------------+---------------+----------------+--------------------------+



In [51]:
sample_date_df.printSchema()

root
 |-- number: long (nullable = true)
 |-- current_data: date (nullable = false)
 |-- current_time: time(6) (nullable = false)
 |-- current_timezone: string (nullable = false)
 |-- current_timestamp: timestamp (nullable = false)



In [ ]:
from pyspark.sql.functions import date_add, date_diff, date_sub, col, lit

(

    sample_date_df
    .withColumn("seven_day_before", date_sub(col("current_date"), 7))
    .withColumn("seven_day_after", date_sub(col("current_date"), 7))
    .withColumn("date_of_birth", lit("1992-07-14").cast("date"))
    .withColumn("date_diff", date_diff(col("current_date"),col("date_of_birth")))
).show()

+------+------------+---------------+----------------+--------------------+----------------+---------------+-------------+---------+
|number|current_data|   current_time|current_timezone|   current_timestamp|seven_day_before|seven_day_after|date_of_birth|date_diff|
+------+------------+---------------+----------------+--------------------+----------------+---------------+-------------+---------+
|     1|  2026-11-08|17:03:18.634472|    Asia/Kolkata|2026-11-08 17:03:...|      2026-11-01|     2026-11-01|   1992-07-14|    12535|
|     2|  2026-11-08|17:03:18.634472|    Asia/Kolkata|2026-11-08 17:03:...|      2026-11-01|     2026-11-01|   1992-07-14|    12535|
|     3|  2026-11-08|17:03:18.634472|    Asia/Kolkata|2026-11-08 17:03:...|      2026-11-01|     2026-11-01|   1992-07-14|    12535|
|     4|  2026-11-08|17:03:18.634472|    Asia/Kolkata|2026-11-08 17:03:...|      2026-11-01|     2026-11-01|   1992-07-14|    12535|
|     5|  2026-11-08|17:03:18.634472|    Asia/Kolkata|2026-11-08 17:0

In [60]:
from pyspark.sql.functions import to_date, col, lit, months_between
(
    sample_date_df
    .withColumn("start", to_date(lit("2016-01-01")))
    .withColumn("end", to_date(lit("2026-01-01")))
    .withColumn("month-between", months_between("start", "end"))
).show()

+------+------------+---------------+----------------+--------------------+----------+----------+-------------+
|number|current_data|   current_time|current_timezone|   current_timestamp|     start|       end|month-between|
+------+------------+---------------+----------------+--------------------+----------+----------+-------------+
|     1|  2026-11-08|17:06:00.678309|    Asia/Kolkata|2026-11-08 17:06:...|2016-01-01|2026-01-01|       -120.0|
|     2|  2026-11-08|17:06:00.678309|    Asia/Kolkata|2026-11-08 17:06:...|2016-01-01|2026-01-01|       -120.0|
|     3|  2026-11-08|17:06:00.678309|    Asia/Kolkata|2026-11-08 17:06:...|2016-01-01|2026-01-01|       -120.0|
|     4|  2026-11-08|17:06:00.678309|    Asia/Kolkata|2026-11-08 17:06:...|2016-01-01|2026-01-01|       -120.0|
|     5|  2026-11-08|17:06:00.678309|    Asia/Kolkata|2026-11-08 17:06:...|2016-01-01|2026-01-01|       -120.0|
+------+------------+---------------+----------------+--------------------+----------+----------+-------

In [62]:
from pyspark.sql.functions import to_date, col, lit, months_between
(
    sample_date_df
    .withColumn("start", to_date(lit("2016-01-01")))
    .withColumn("end", to_date(lit("2026-01-01")))
    .withColumn("month-between", months_between("start", "end"))
).show()

+------+------------+---------------+----------------+--------------------+----------+----------+-------------+
|number|current_data|   current_time|current_timezone|   current_timestamp|     start|       end|month-between|
+------+------------+---------------+----------------+--------------------+----------+----------+-------------+
|     1|  2026-11-08|17:07:15.763991|    Asia/Kolkata|2026-11-08 17:07:...|2016-01-01|2026-01-01|       -120.0|
|     2|  2026-11-08|17:07:15.763991|    Asia/Kolkata|2026-11-08 17:07:...|2016-01-01|2026-01-01|       -120.0|
|     3|  2026-11-08|17:07:15.763991|    Asia/Kolkata|2026-11-08 17:07:...|2016-01-01|2026-01-01|       -120.0|
|     4|  2026-11-08|17:07:15.763991|    Asia/Kolkata|2026-11-08 17:07:...|2016-01-01|2026-01-01|       -120.0|
|     5|  2026-11-08|17:07:15.763991|    Asia/Kolkata|2026-11-08 17:07:...|2016-01-01|2026-01-01|       -120.0|
+------+------------+---------------+----------------+--------------------+----------+----------+-------

In [72]:
from pyspark.sql.functions import unix_timestamp , try_to_timestamp, current_timestamp

sample_date_df.select(try_to_timestamp(lit("2016-20-12")),to_date(lit("2017-12-11"))).show(1)

+----------------------------+-------------------+
|try_to_timestamp(2016-20-12)|to_date(2017-12-11)|
+----------------------------+-------------------+
|                        NULL|         2017-12-11|
+----------------------------+-------------------+
only showing top 1 row


In [ ]:
dateFormat = "yyyy-dd-MM"
from pyspark.sql.functions import to_timestamp

(
    sample_date_df
    .withColumn("formatedDate", to_date(lit("2027-12-11"), dateFormat))
    .withColumn("formatedDate1", to_date(lit("2027-11-12"), "yyyy-MM-dd"))
    .withColumn("formatedDate2", to_timestamp(lit("2027-11-12"), "yyyy-MM-dd"))
    .filter(col("formatedDate1") > lit("2026-10-12"))   ## right date formate can be compare like this
).show()

+------+------------+---------------+----------------+--------------------+------------+-------------+-------------------+
|number|current_data|   current_time|current_timezone|   current_timestamp|formatedDate|formatedDate1|      formatedDate2|
+------+------------+---------------+----------------+--------------------+------------+-------------+-------------------+
|     1|  2026-11-08|17:15:56.058052|    Asia/Kolkata|2026-11-08 17:15:...|  2027-11-12|   2027-11-12|2027-11-12 00:00:00|
|     2|  2026-11-08|17:15:56.058052|    Asia/Kolkata|2026-11-08 17:15:...|  2027-11-12|   2027-11-12|2027-11-12 00:00:00|
|     3|  2026-11-08|17:15:56.058052|    Asia/Kolkata|2026-11-08 17:15:...|  2027-11-12|   2027-11-12|2027-11-12 00:00:00|
|     4|  2026-11-08|17:15:56.058052|    Asia/Kolkata|2026-11-08 17:15:...|  2027-11-12|   2027-11-12|2027-11-12 00:00:00|
|     5|  2026-11-08|17:15:56.058052|    Asia/Kolkata|2026-11-08 17:15:...|  2027-11-12|   2027-11-12|2027-11-12 00:00:00|
+------+--------

In [ ]:
spark.stop()